In [2]:
!pip install chromadb pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 109.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 100.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 43.5 MB/s  0:00:00
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.5.0
    Uninstalling urllib3-2.5.0:
      Successfully uninstalled urllib3-2.5.0
  Attempting uninstall: python-dateutil
    Found existing installation: python-dateutil 2.9.0.post0
    Uninstalling python-dateutil-2.9.0.post0:
      Successfully uninstalled python-dateutil-2.9.0.post0
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.42.1
    Uninstalling opentelemetry-api-1.42.1:
      Successfully uninstalled opentelemetry-api-1.42.1
  Attempting uninstall: oauthlib
    Found existing installation: oauthlib 3.3.1
    Uninstalling oauthlib-3.3.1:
      Successfully uninstalled oauthlib-3.3.1
  Attempting uninstall: opentelemetry-semantic-conv

In [1]:
# import imp libraries

import chromadb
from pypdf import PdfReader
from google.colab import files, drive
from sentence_transformers import SentenceTransformer


In [3]:
upload = files.upload()
print ("Total files uploaded - ",len(upload))

Saving 01_Bank_Overview_and_Account_Types.pdf to 01_Bank_Overview_and_Account_Types (1).pdf
Saving 02_Savings_Account_Policy.pdf to 02_Savings_Account_Policy (1).pdf
Saving 03_Current_Account_Policy.pdf to 03_Current_Account_Policy.pdf
Saving 04_Personal_Loan_Policy.pdf to 04_Personal_Loan_Policy.pdf
Saving 05_Credit_Card_Policy.pdf to 05_Credit_Card_Policy.pdf
Saving 06_Fixed_Deposit_Policy.pdf to 06_Fixed_Deposit_Policy.pdf
Saving 07_Digital_Banking_and_Security.pdf to 07_Digital_Banking_and_Security.pdf
Saving 08_Service_Charges_and_Fees.pdf to 08_Service_Charges_and_Fees.pdf
Total files uploaded -  8


In [6]:
all_documents = []

for filename in upload.keys():
  reader = PdfReader(filename)

  for page_no, page in enumerate(reader.pages):
    text = page.extract_text()
    if text and text.strip():
      all_documents.append({
          "text" :text,
          "source": filename,
          "page": page_no + 1

        }  )

print ("Total documents - ", len(all_documents))

Total documents -  16


In [7]:
#Perform Text Chunking

def chunk_text(text, chunk_size=500):

    chunks = []

    for i in range(0, len(text), chunk_size):

        chunks.append(text[i:i+chunk_size])

    return chunks

In [8]:
# Process all documents

all_chunks = []

for doc in all_documents:

    chunks = chunk_text(doc["text"])

    for chunk in chunks:

        all_chunks.append({
            "text": chunk,
            "source": doc["source"],
            "page": doc["page"]
        })

print("Total chunks:", len(all_chunks))

Total chunks: 70


In [9]:
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

... (3 outputs left out)

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [10]:
texts = [chunk["text"] for chunk in all_chunks]

In [11]:
embeddings = model.encode(texts).tolist()

In [12]:
print(len(embeddings))

70


In [13]:
drive.mount('/content/drive')

Mounted at /content/drive


In [14]:
path = "/content/drive/MyDrive/RAG and VectorDB - St. John/DAY 4/bank_database"

In [15]:
client = chromadb.PersistentClient(path=path)

In [16]:
collection = client.get_or_create_collection(name="bank_collection")

In [17]:
collection.add(

    ids=[f"chunk_{i}" for i in range(len(all_chunks))],

    documents=texts,

    embeddings=embeddings,

    metadatas=[
        {
            "source": chunk["source"],
            "page": chunk["page"]
        }
        for chunk in all_chunks
    ]
)

In [18]:
print("Total stored chunks:", collection.count())


Total stored chunks: 70


In [19]:
query = input (" Enter your query :")

 Enter your query :tell me about personal loan


In [20]:
query_embedding = model.encode(query).tolist()

In [23]:
results = collection.query(

    query_embeddings=query_embedding,

    n_results=3,

    include=["metadatas", "documents", "distances"]

)

In [28]:
context = ""

for i in range(len(results["documents"][0])):
  document = results["documents"][0][i]
  metadata =results["metadatas"][0][i]

  context += document + "/n/n"

  print("retrived document ",i+1)
  print("document :",document)
  print(f"Source : {metadata['source']}")
  print(f"Page : {metadata['page']}")

retrived document  1
document : cation after assessment.
Interest, fees and repayment
The illustrative personal-loan interest rate for this dataset is 12.5% per annum on a reducing-balance basis. The
sample processing fee is 2% of the sanctioned loan amount, subject to a minimum of ■1,000 plus applicable taxes.
Repayment is made through monthly instalments (EMIs) on the due date stated in the sanction letter. The sample
late-payment charge is ■500 plus applicable taxes for an overdue instalment. Actual charges in a real loan w
Source : 04_Personal_Loan_Policy.pdf
Page : 1
retrived document  2
document : rsonal Loan is a fictional unsecured loan intended for eligible individuals who need funds for personal
purposes. The sample loan amount ranges from ■50,000 to ■10,00,000, with a sample repayment tenure of 12 to 60
months.
Applicants must be at least 21 years old and no older than 60 years at loan maturity. Salaried applicants should have
at least two years of total work experience and 

In [29]:
chat_history = []

In [30]:
from google import genai
from google.colab import userdata

In [31]:
client = genai.Client(
    api_key=userdata.get("GEMINI_API_KEY")
)

In [33]:
while True:
  query = input("Enter your query :")

  if query.lower() == "exit":
    break

  query_embedding = model.encode(query).tolist()

  results = collection.query(

    query_embeddings=query_embedding,

    n_results=3,

    include=["metadatas", "documents"])

  # preparing data
  context = ""
  for documents in results["documents"][0]:
    context += documents + "/n/n"

  # converting previous chat into text
  history_text = ""

  for msg in chat_history:
    history_text = (
            message["role"] + ": " +
            message["content"] + "\n"
        )


  prompt = f"""
            You are an AI powered customer chatbot for a bank.

            your responsibility is to assist customer by answering question related to
            bank product and services.

            Answer only using given context. If answer is not available in context
            say Sorry! i wont be able to help you with that.

            Previous conversation : {history_text}

            Context : {context}

            Question : {query}

            answer:

  """

  response = client.models.generate_content(
      model = "gemini-3.5-flash-lite",
      contents = prompt
  )

  answer = response.text

  print("\n BOT :", answer)

  chat_history.append({"role": "user", "content": query})
  chat_history.append({"role": "assistant", "content": answer})



"""
questions to ask bot
1. tell me about personal loan
2. tell me about interest rate
3. tell me about maximum repayment periods
4. tell me about minimum balance required
"""


Enter your query :tell me about personal loan

 BOT : Based on the provided context, the personal loan is a fictional unsecured loan intended for eligible individuals who need funds for personal purposes. 

Here are the key details regarding the personal loan:
* **Loan Amount:** Ranges from ■50,000 to ■10,00,000.
* **Repayment Tenure:** 12 to 60 months, made through monthly instalments (EMIs) on the due date.
* **Interest Rate:** Illustrative interest rate is 12.5% per annum on a reducing-balance basis.
* **Processing Fee:** Sample processing fee is 2% of the sanctioned loan amount, subject to a minimum of ■1,000 plus applicable taxes.
* **Late-Payment Charge:** Sample late-payment charge is ■500 plus applicable taxes for an overdue instalment.
* **Eligibility Criteria:** 
  * Applicants must be between 21 years old and no older than 60 years at loan maturity.
  * Salaried applicants need at least two years of total work experience and six months with their current employer, with a min

KeyboardInterrupt: Interrupted by user

In [34]:
chat_history_new = []

In [42]:
while True:
  query = input("Enter your query :")

  if query.lower() == "exit":
    break

  # history text

  history_text = ""

  for msg in chat_history_new:
    history_text = (
            msg["role"] + ": " +
            msg["content"] + "\n"
        )

 # rewrtiing prompt
  prompt = f"""
              You are a question rewriting assistant.

    Based on the conversation history,
    rewrite the latest question into a
    complete standalone question.

    Do not answer the question.
    Return only the rewritten question.

    Conversation History:
    {history_text}

    Latest Question:
    {query}

    Rewritten Question:
    """
  rewritten_question = client.models.generate_content(
      model = "gemini-3.5-flash-lite",
      contents = prompt
  )

  rewritten_query = rewritten_question.text.strip()

  print("\n Rewritten Question :", rewritten_query)

  query_embedding = model.encode(rewritten_query).tolist()

  results = collection.query(

    query_embeddings=query_embedding,

    n_results=3,

    include=["metadatas", "documents"])

  context = ""
  for documents in results["documents"][0]:
    context += documents + "/n/n"

  prompt = f"""
            You are an AI-powered customer support assistant for ABC Bank.

    Your responsibility is to assist customers by answering
    questions related to the bank's products, services, policies, and procedures
    If the answer is not available,
    say that you do not have that information.

    Retrieved Context:
    {context}

    Question:
    {rewritten_query}

    Answer:
    """
  response = client.models.generate_content(
      model = "gemini-3.5-flash-lite",
      contents = prompt
  )

  answer = response.text
  print("\n BOT :", answer)

# STEP 14: Save conversation

  chat_history_new.append({
        "role": "User",
        "content": query
    })

  chat_history_new.append({
        "role": "Assistant",
        "content": answer
    })



Enter your query :tell me about personal loan

 Rewritten Question : Can you provide information about personal loans?

 BOT : Based on the provided information regarding loan applications, employed applicants may be asked to provide identity and address proof, a PAN or applicable declaration, business proof, recent bank statements, income-tax returns or financial statements, and details of existing loans. The bank may also request additional documents to verify income, obligations, and the purpose of the loan. 

For the application journey, the customer submits an application and supporting documents, the bank completes verification and credit assessment, communicates the decision, and—if approved—issues the loan. Additionally, for income and credit assessment, the sample minimum monthly net income for salaried applicants is ₹25,000 (note: text displays as ■25,000 in source), and the bank reviews income stability, existing obligations, repayment history, credit information, the reques